In [6]:
import numpy as np
import mpi4py.MPI as MPI
import dolfinx
from dolfinx.io import gmsh
import ufl
from petsc4py import PETSc

data_dir = "../data/"

mesh_data = gmsh.read_from_msh(
    f"{data_dir}layered_channel.msh",
    MPI.COMM_WORLD,
    rank=0,
    gdim=3,
)

mesh = mesh_data.mesh
cell_tags = mesh_data.cell_tags
facet_tags = mesh_data.facet_tags

Info    : Reading '../data/layered_channel.msh'...
Info    : 45 entities
Info    : 15193 nodes
Info    : 65091 elements
Info    : Done reading '../data/layered_channel.msh'


In [7]:
# ============================================================
# 2. PHYSICAL PARAMETERS
# ============================================================

# Fluid properties
rho = 1000.0       # kg/m^3
mu = 1e-3          # Pa*s

# Channel dimensions
L = 58.5e-3        # m
W_channel = 2.5e-3 # m
H_tot = 0.35e-3    # m
H_gel = 0.10e-3    # m
H_fluid = 0.25e-3  # m

# Flow rate
Q = 1e-9 / 60      # 1 uL/min -> m^3/s

# Mean inlet velocity
A_inlet = W_channel * H_fluid
u_avg = Q / A_inlet

print(f"Mean inlet velocity = {u_avg:.3e} m/s")

Mean inlet velocity = 2.667e-05 m/s


In [8]:
# ============================================================
# 3. TAYLOR-HOOD VELOCITY/PRESSURE SPACE
# ============================================================

# P2 vector-valued velocity element
v_element = basix.ufl.element(
    "Lagrange",
    mesh.basix_cell(),
    2,
    shape=(mesh.geometry.dim,),
)

# P1 pressure element
p_element = basix.ufl.element(
    "Lagrange",
    mesh.basix_cell(),
    1,
)

# Mixed Taylor-Hood element
th_element = basix.ufl.mixed_element(
    [v_element, p_element]
)

W = dolfinx.fem.functionspace(
    mesh,
    th_element,
)

u, p = ufl.TrialFunctions(W)
v, q = ufl.TestFunctions(W)

In [9]:
# ============================================================
# 4. FLOW BOUNDARIES
#
# Physical groups from the Gmsh file:
#
#   101 = CellLayer
#   102 = Inlet
#   103 = Outlet
#
# NOTE:
# The original Gmsh script does NOT create physical groups
# for the top/side walls, so those walls cannot be identified
# by facet tag here.
# ============================================================

tdim = mesh.topology.dim
fdim = tdim - 1

mesh.topology.create_connectivity(fdim, tdim)

In [10]:
# ------------------------------------------------------------
# Inlet
# ------------------------------------------------------------

inlet_facets = facet_tags.find(102)

inlet_dofs = dolfinx.fem.locate_dofs_topological(
    W.sub(0),
    fdim,
    inlet_facets,
)


# Prescribed inlet velocity
u_inlet = dolfinx.fem.Function(W.sub(0))

u_inlet.interpolate(
    lambda x: np.vstack(
        (
            np.full(x.shape[1], u_avg),
            np.zeros(x.shape[1]),
            np.zeros(x.shape[1]),
        )
    )
)

bc_inlet = dolfinx.fem.dirichletbc(
    u_inlet,
    inlet_dofs,
)


RuntimeError: Cannot create Function from subspace. Consider collapsing the function space